# Request lifecycle and orchestration

## Learning goals
- Follow a request through the orchestrator state machine
- See one public request fan out into per-stage requests
- Distinguish inter-stage outputs from final client outputs

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## Lifecycle

```text
HTTP/offline request
 -> normalize input
 -> AsyncOmniEngine
 -> Orchestrator creates request state
 -> stage 0 engine -> output processor -> connector
 -> stage 1 engine -> ... -> final OmniRequestOutput / stream
```

The orchestrator is a control-plane component: it schedules stages and routes data, but does not execute neural layers itself. Each stage has its own engine, scheduler, and KV manager.

In [ ]:
STATES = ['RECEIVED', 'STAGE_0_RUNNING', 'TRANSFER_0_1', 'STAGE_1_RUNNING',
          'TRANSFER_1_2', 'STAGE_2_RUNNING', 'COMPLETED']
for i, s in enumerate(STATES):
    print(f'{i}  {s}')

## Request state: one public id, many stage ids

A single client request can spawn several stage requests (and companion branches). The orchestrator tracks their ids, dependencies, streaming status, cancellation, and which outputs are *final* vs *internal*.

In [ ]:
def run_lifecycle(public_id, n_stages, cancel_at=None):
    req = {'public_id': public_id, 'stage_ids': {}, 'status': 'RECEIVED',
           'final_outputs': []}
    for stage in range(n_stages):
        if cancel_at == stage:
            req['status'] = f'CANCELLED_AT_STAGE_{stage}'
            return req
        req['stage_ids'][stage] = f'{public_id}-stage-{stage}'
        req['status'] = f'STAGE_{stage}_RUNNING'
    req['status'] = 'COMPLETED'
    req['final_outputs'] = ['audio.wav']  # only the last stage is client-facing
    return req
print(run_lifecycle('req-7', 3))
print(run_lifecycle('req-8', 3, cancel_at=1))

## Why streaming decouples the stages

The output processor asynchronously streams partial output to the next stage, so the Talker can start before the Thinker fully finishes. This is what reduces time-to-first-token.

In [ ]:
# toy: thinker emits tokens; talker consumes them as they arrive
thinker_stream = iter(['The', 'cat', 'sat', 'down'])
for i, tok in enumerate(thinker_stream):
    print(f't={i}: thinker emitted {tok!r} -> talker can begin decoding on it')

## Exercise

Modify `run_lifecycle` to record a timestamp per state transition (use a simple counter). Which transition would you watch to measure time-to-first-token?

In [ ]:
# Solution: TTFT is the time from RECEIVED to the FIRST client-visible token,
# which (with streaming) happens as soon as the LAST stage emits its first output,
# not when the whole pipeline COMPLETES.
print('Watch: RECEIVED -> first emit from the final (client-facing) stage.')

## Source lab

Read `entrypoints/omni.py` -> `engine/async_omni_engine.py` -> `engine/orchestrator.py`. Find request-state creation, stage submission, output consumption, and cancellation.